# Cholesterol Imputation

In [37]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

In [24]:
df = pd.read_csv("data/diabetes_012_health_indicators_BRFSS2015.csv")

In [25]:
# Add BMI Age interaction
df["BMI_Age"] = df["BMI"] * df["Age"]

In [ ]:
# Find accuracy of mode imputation for HighChol
df["HighChol"].value_counts(normalize=True)

HighChol
0.0    0.575879
1.0    0.424121
Name: proportion, dtype: float64

In [26]:
# Prepare X and Y variables

# Drop Diabetes_012 since it won't be known at prediction time
chol_df = df.drop(columns=["Diabetes_012"])

X = chol_df.drop(columns=["HighChol"])
Y = chol_df["HighChol"]

In [9]:
# Create column transformer (Categorical features included in case come are engineered later)
num_features = X.select_dtypes(exclude=["object"]).columns
cat_features = X.select_dtypes(include=["object"]).columns

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features),
    ]
)

In [10]:
X = preprocessor.fit_transform(X)

In [ ]:
# Train test split
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

In [14]:
# Evaluation function
def evaluate_model(true, predicted, proba):
    accuracy = accuracy_score(true, predicted)
    precision = precision_score(true, predicted)
    recall = recall_score(true, predicted)
    auc = roc_auc_score(true, proba)

    return accuracy, precision, recall, auc

In [17]:
# Logistic Regression Baseline
baseline_model = LogisticRegression(random_state=42)
baseline_model.fit(x_train, y_train)

y_pred = baseline_model.predict(x_test)
y_proba = baseline_model.predict_proba(x_test)[:, 1]

accuracy, precision, recall, auc = evaluate_model(y_test, y_pred, y_proba)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"AUC: {auc:.4f}")

Accuracy: 0.6715
Precision: 0.6332
Recall: 0.5357
AUC: 0.7268


In [20]:
# Testing all models
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0),
    "LightGBM": LGBMClassifier(random_state=42)
}

model_list = []
accuracy_list = []
precision_list = []
recall_list = []
auc_list = []

for i in range(len(list(models))):

    # Train model
    model = list(models.values())[i]

    model.fit(x_train, y_train)

    # Predict
    y_pred = model.predict(x_test)
    y_proba = model.predict_proba(x_test)[:, 1]

    # Evaluate
    accuracy, precision, recall, auc = evaluate_model(y_test, y_pred, y_proba)

    print(list(models.keys())[i])
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"AUC: {auc:.4f}")

    model_list.append(list(models.keys())[i])
    accuracy_list.append(accuracy)
    precision_list.append(precision)
    recall_list.append(recall)
    auc_list.append(auc)

    print("-" * 35)
    print("\n")

Logistic Regression
Accuracy: 0.6715
Precision: 0.6332
Recall: 0.5357
AUC: 0.7268
-----------------------------------


Random Forest
Accuracy: 0.6527
Precision: 0.5973
Recall: 0.5562
AUC: 0.6989
-----------------------------------


XGBoost
Accuracy: 0.6768
Precision: 0.6331
Recall: 0.5655
AUC: 0.7368
-----------------------------------


CatBoost
Accuracy: 0.6777
Precision: 0.6340
Recall: 0.5679
AUC: 0.7382
-----------------------------------


[LightGBM] [Info] Number of positive: 86073, number of negative: 116871
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020538 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 473
[LightGBM] [Info] Number of data points in the train set: 202944, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.424122 -> initscore=-0.305875
[LightGBM] [Info] Start training from score 

e:\Projects\Diabetes Risk Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
e:\Projects\Diabetes Risk Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM
Accuracy: 0.6795
Precision: 0.6368
Recall: 0.5685
AUC: 0.7397
-----------------------------------




## Compare Gradient Boosting Models to Mode Imputation

In [36]:
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0),
    "LightGBM": LGBMClassifier(random_state=42)
}

model_list = []
accuracy_list = []
precision_list = []
recall_list = []
auc_list = []

# Create missing data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["HighChol"])
y_true = test_df["HighChol"].copy()

mask = np.random.rand(len(test_df)) < 0.2
test_missing = test_df.copy()
test_missing.loc[mask, "HighChol"] = np.nan

mode = train_df["HighChol"].mode()[0]
test_missing["HighChol"] = test_missing["HighChol"].fillna(mode)

y_eval_true = y_true[mask]
y_eval_pred = test_missing.loc[mask, "HighChol"]

positive_rate = train_df["HighChol"].mean()
y_eval_proba = np.full(len(y_eval_true), positive_rate)

accuracy, precision, recall, auc = evaluate_model(y_eval_true, y_eval_pred, y_eval_proba)
model_list.append("Mode Imputation")
accuracy_list.append(accuracy)
precision_list.append(precision)
recall_list.append(recall)
auc_list.append(auc)

print("Mode Imputation Evaluation")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"AUC: {auc:.4f}")

print("-" * 35)
print("\n")

for i in range(len(list(models))):

    # Train model
    model = list(models.values())[i]

    model.fit(train_df.drop(columns=["HighChol"]), train_df["HighChol"])

    # Predict
    y_pred = model.predict(test_df.drop(columns=["HighChol"]))
    y_proba = model.predict_proba(test_df.drop(columns=["HighChol"]))[:, 1]

    # Evaluate
    accuracy, precision, recall, auc = evaluate_model(y_test, y_pred, y_proba)

    print(list(models.keys())[i])
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"AUC: {auc:.4f}")

    model_list.append(list(models.keys())[i])
    accuracy_list.append(accuracy)
    precision_list.append(precision)
    recall_list.append(recall)
    auc_list.append(auc)

    print("-" * 35)
    print("\n")

e:\Projects\Diabetes Risk Prediction\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Mode Imputation Evaluation
Accuracy: 0.5776
Precision: 0.0000
Recall: 0.0000
AUC: 0.5000
-----------------------------------




e:\Projects\Diabetes Risk Prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression
Accuracy: 0.6707
Precision: 0.6386
Recall: 0.5151
AUC: 0.7217
-----------------------------------


Random Forest
Accuracy: 0.6527
Precision: 0.5975
Recall: 0.5552
AUC: 0.6988
-----------------------------------


XGBoost
Accuracy: 0.6768
Precision: 0.6331
Recall: 0.5655
AUC: 0.7368
-----------------------------------


CatBoost
Accuracy: 0.6777
Precision: 0.6340
Recall: 0.5679
AUC: 0.7382
-----------------------------------


[LightGBM] [Info] Number of positive: 86073, number of negative: 116871
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014033 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 457
[LightGBM] [Info] Number of data points in the train set: 202944, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.424122 -> initscore=-0.305875
[LightGBM] [Info] Start training from score 

## CatBoost Hyperparameter Tuning

In [38]:
# Create validation sets
x_train_part, x_val, y_train_part, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

In [ ]:
cat_param_dist = {
    "depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "iterations": [300, 600, 1000]
}

tuning_model = CatBoostClassifier(random_state=42, verbose=0, loss_function="Logloss", auto_class_weights="Balanced")
tuning_model.fit(x_train, y_train)

random_search = RandomizedSearchCV(tuning_model, cat_param_dist, n_iter=20, scoring="roc_auc", cv=3, n_jobs=-1, random_state=42)
random_search.fit(x_train_part, y_train_part, eval_set=(x_val, y_val), early_stopping_rounds=10)

tune_pred = random_search.predict(x_test)
tune_proba = random_search.predict_proba(x_test)[:, 1]

accuracy, precision, recall, auc = evaluate_model(y_test, tune_pred, tune_proba)

print("Tuned CatBoost Evaluation")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"AUC: {auc:.4f}")

Tuned CatBoost Evaluation
Accuracy: 0.6731
Precision: 0.5978
Recall: 0.7003
AUC: 0.7403


In [40]:
print("Best Hyperparameters:", random_search.best_params_)

Best Hyperparameters: {'learning_rate': 0.05, 'iterations': 600, 'depth': 6}
